# 05 - Sentence-BERT Semantic Retrieval

Uses `all-MiniLM-L6-v2` sentence embeddings and cosine similarity to rank WTO case summaries, followed by evaluation on three legal queries.

**Expected input:** `data/processed/preprocessed_wto_cases.csv`

> Install dependencies once from the repository root with `pip install -r requirements.txt`. Run the notebooks in numerical order.

In [ ]:
from pathlib import Path

# Resolve repository paths whether Jupyter starts in the repository root or notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_PDF_DIR = DATA_DIR / "raw" / "wto_case_pdfs"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

RAW_PDF_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = PROCESSED_DIR / "wto_cases.csv"
PROCESSED_DATA_PATH = PROCESSED_DIR / "preprocessed_wto_cases.csv"

print(f"Project root: {PROJECT_ROOT}")


In [2]:
import pandas as pd

# Load the cleaned and pre-processed dataset 
df = pd.read_csv(PROCESSED_DATA_PATH)

# Display the first few rows
df.head()

,Case ID,Title,Complainant,Respondent,Date,Articles,Summary,Title_normalized,Summary_normalized,Articles_normalized,Title_tokenized,Summary_tokenized,Articles_tokenized,Summary_lemmatized,Articles_lemmatized,Summary_segmented
0,ds491,US – COATED PAPER (INDONESIA)1,Indonesia,United States,22 January 2018,"ADA Arts. 3.5, 3.7, 3.8;\nASCM Arts. 2.1(c), 1...",FINDINGS2\n• ASCM Art. 14(d) (rejection of in-...,us coated paper indonesia1,findings2\n ascm art 14d rejection of incountr...,ada arts 35 37 38\nascm arts 21c 127 14d 155\n...,us coated paper indonesia1,findings2 ascm art 14d rejection incountry pri...,ada arts 35 37 38 ascm arts 21c 127 14d 155 15...,findings2 ascm art 14d rejection incountry pri...,ada arts 35 37 38 ascm art 21c 127 14d 155 157...,['findings2 ascm art 14d rejection incountry p...
1,ds33,US – WOOL SHIRTS AND BLOUSES1,India,United States,23 May 1997,ATC Arts. 6 and 2.4,/AB FINDINGS\n• ATC Art. 6 (transitional safeg...,us wool shirts and blouses1,ab findings\n atc art 6 transitional safeguard...,atc arts 6 and 24,us wool shirts blouses1,ab findings atc art 6 transitional safeguard m...,atc arts 6 24,ab finding atc art 6 transitional safeguard me...,atc arts 6 24,['ab finding atc art 6 transitional safeguard ...
2,ds403,PHILIPPINES – DISTILLED SPIRITS1,"European Union,\nUnited States",Philippines,20 January 2012,"GATT Art. III:2, first and\nsecond sentences",/AB FINDINGS\n• GATT Art. III:2 (national trea...,philippines distilled spirits1,ab findings\n gatt art iii2 national treatment...,gatt art iii2 first and\nsecond sentences,philippines distilled spirits1,ab findings gatt art iii2 national treatment t...,gatt art iii2 first second sentences,ab findings gatt art iii2 national treatment t...,gatt art iii2 first second sentence,['ab findings gatt art iii2 national treatment...
3,ds267A,US – UPLAND COTTON (ARTICLE 21.5 – BRAZIL)1,Brazil,United States,20 June 2008,"ASCM Arts.3, 5(c), 6.3(c), and item\n(j) of th...","/AB FINDINGS\n• AA Arts. 10.1 and 8, and ASCM ...",us upland cotton article 215 brazil1,ab findings\n aa arts 101 and 8 and ascm arts ...,ascm arts3 5c 63c and item\nj of the illustrat...,us upland cotton article 215 brazil1,ab findings aa arts 101 8 ascm arts 31a 32 ill...,ascm arts3 5c 63c item j illustrative list aa ...,ab findings aa arts 101 8 ascm art 31a 32 illu...,ascm arts3 5c 63c item j illustrative list aa ...,['ab findings aa arts 101 8 ascm art 31a 32 il...
4,ds44,JAPAN – FILM1,United States,Japan,22 April 1998,"GATT Arts. XXIII:1(b), III:4 and X:1",FINDINGS\n• GATT Art. XXIII:1(b) (non-violatio...,japan film1,findings\n gatt art xxiii1b nonviolation claim...,gatt arts xxiii1b iii4 and x1,japan film1,findings gatt art xxiii1b nonviolation claim p...,gatt arts xxiii1b iii4 x1,findings gatt art xxiii1b nonviolation claim p...,gatt arts xxiii1b iii4 x1,['findings gatt art xxiii1b nonviolation claim...


In [3]:
df['text_for_retrieval'] = (
    df['Title_tokenized'].fillna('') + ' ' +
    df['Articles_lemmatized'].fillna('') + ' ' +
    df['Summary_lemmatized'].fillna('')
)

In [7]:
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import precision_score, recall_score
import numpy as np

# Load the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all cases
corpus_embeddings = model.encode(df['text_for_retrieval'].tolist(), convert_to_tensor=True)

# Encode query
query = "US - China Case Laws using Article VIII of GATT"
query_embedding = model.encode(query, convert_to_tensor=True)

# Compute cosine similarity and get top 5 indices
top_results = util.cos_sim(query_embedding, corpus_embeddings).topk(k=5)
top_indices = top_results[1][0]  # shape [1, 5]

# Extract top 5 SBERT results (Case ID + Title only)
sbert_top_5 = []
for idx in top_indices:
    idx = idx.item()
    case_id = df.iloc[idx]['Case ID']
    title = df.iloc[idx]['Title']
    sbert_top_5.append((case_id, title))

# Display top 5 results
print("SBERT Top 5 Results:")
for i, (case_id, title) in enumerate(sbert_top_5, 1):
    print(f"{i}. {case_id} – {title}")

# ------------------------------------------
# 📊 Evaluation against ground truth
# ------------------------------------------
ground_truth_ids = {"ds437", "ds440", "ds437A", "ds517", "ds395"}  

# Binary relevance vector for evaluation
predicted_ids = [case_id for case_id, _ in sbert_top_5]
relevance = [1 if case_id in ground_truth_ids else 0 for case_id in predicted_ids]

# Precision@5
precision_at_5 = sum(relevance) / len(relevance)

# Recall@5
recall_at_5 = sum(relevance) / len(ground_truth_ids)

# nDCG@5 calculation
def dcg(rel):
    return sum(r / np.log2(i + 2) for i, r in enumerate(rel))

def ndcg(rel):
    ideal_rel = sorted(rel, reverse=True)
    return dcg(rel) / dcg(ideal_rel) if dcg(ideal_rel) != 0 else 0.0

ndcg_at_5 = ndcg(relevance)

# Print evaluation results
print("\n SBERT Evaluation Metrics:")
print(f"Precision@5: {precision_at_5:.4f}")
print(f"Recall@5: {recall_at_5:.4f}")
print(f"nDCG@5: {ndcg_at_5:.4f}")


SBERT Top 5 Results:
1. ds395 – CHINA – RAW MATERIALS1
2. ds432 – CHINA – RARE EARTHS1
3. ds27B – EC – BANANAS III (ARTICLE 21.5 – US)1
4. ds517 – CHINA – TRQS1
5. ds510 – US – RENEWABLE ENERGY (INDIA)1

 SBERT Evaluation Metrics:
Precision@5: 0.4000
Recall@5: 0.4000
nDCG@5: 0.8772


In [7]:
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import precision_score, recall_score
import numpy as np

# Load the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all cases
corpus_embeddings = model.encode(df['text_for_retrieval'].tolist(), convert_to_tensor=True)

# Encode query
query = "Case Laws involving European Communities as Complainant and United States as Respondent"
query_embedding = model.encode(query, convert_to_tensor=True)

# Compute cosine similarity and get top 5 indices
top_results = util.cos_sim(query_embedding, corpus_embeddings).topk(k=5)
top_indices = top_results[1][0]  # shape [1, 5]

# Extract top 5 SBERT results (Case ID + Title only)
sbert_top_5 = []
for idx in top_indices:
    idx = idx.item()
    case_id = df.iloc[idx]['Case ID']
    title = df.iloc[idx]['Title']
    sbert_top_5.append((case_id, title))

# Display top 5 results
print("SBERT Top 5 Results:")
for i, (case_id, title) in enumerate(sbert_top_5, 1):
    print(f"{i}. {case_id} – {title}")

# ------------------------------------------
# 📊 Evaluation against ground truth
# ------------------------------------------
ground_truth_ids = {"ds353A", "ds108A", "ds165", "ds34", "ds108B"}  

# Binary relevance vector for evaluation
predicted_ids = [case_id for case_id, _ in sbert_top_5]
relevance = [1 if case_id in ground_truth_ids else 0 for case_id in predicted_ids]

# Precision@5
precision_at_5 = sum(relevance) / len(relevance)

# Recall@5
recall_at_5 = sum(relevance) / len(ground_truth_ids)

# nDCG@5 calculation
def dcg(rel):
    return sum(r / np.log2(i + 2) for i, r in enumerate(rel))

def ndcg(rel):
    ideal_rel = sorted(rel, reverse=True)
    return dcg(rel) / dcg(ideal_rel) if dcg(ideal_rel) != 0 else 0.0

ndcg_at_5 = ndcg(relevance)

# Print evaluation results
print("\n SBERT Evaluation Metrics:")
print(f"Precision@5: {precision_at_5:.4f}")
print(f"Recall@5: {recall_at_5:.4f}")
print(f"nDCG@5: {ndcg_at_5:.4f}")


SBERT Top 5 Results:
1. ds62 – EC – COMPUTER EQUIPMENT1
2. ds108B – US – FSC (ARTICLE 21.5 – EC II)1
3. ds231 – EC – SARDINES1
4. ds577 – US — RIPE OLIVES FROM SPAIN1
5. ds34 – TURKEY – TEXTILES1

 SBERT Evaluation Metrics:
Precision@5: 0.4000
Recall@5: 0.4000
nDCG@5: 0.6241


In [5]:
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import precision_score, recall_score
import numpy as np

# Load the model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all cases
corpus_embeddings = model.encode(df['text_for_retrieval'].tolist(), convert_to_tensor=True)

# Encode query
query = "Case Laws involving Australia with ASCM Article Application"
query_embedding = model.encode(query, convert_to_tensor=True)

# Compute cosine similarity and get top 5 indices
top_results = util.cos_sim(query_embedding, corpus_embeddings).topk(k=5)
top_indices = top_results[1][0]  # shape [1, 5]

# Extract top 5 SBERT results (Case ID + Title only)
sbert_top_5 = []
for idx in top_indices:
    idx = idx.item()
    case_id = df.iloc[idx]['Case ID']
    title = df.iloc[idx]['Title']
    sbert_top_5.append((case_id, title))

# Display top 5 results
print("SBERT Top 5 Results:")
for i, (case_id, title) in enumerate(sbert_top_5, 1):
    print(f"{i}. {case_id} – {title}")

# ------------------------------------------
# 📊 Evaluation against ground truth
# ------------------------------------------
ground_truth_ids = {"ds126A", "ds126", "ds18A", "ds367", "ds529"}  

# Binary relevance vector for evaluation
predicted_ids = [case_id for case_id, _ in sbert_top_5]
relevance = [1 if case_id in ground_truth_ids else 0 for case_id in predicted_ids]

# Precision@5
precision_at_5 = sum(relevance) / len(relevance)

# Recall@5
recall_at_5 = sum(relevance) / len(ground_truth_ids)

# nDCG@5 calculation
def dcg(rel):
    return sum(r / np.log2(i + 2) for i, r in enumerate(rel))

def ndcg(rel):
    ideal_rel = sorted(rel, reverse=True)
    return dcg(rel) / dcg(ideal_rel) if dcg(ideal_rel) != 0 else 0.0

ndcg_at_5 = ndcg(relevance)

# Print evaluation results
print("\n SBERT Evaluation Metrics:")
print(f"Precision@5: {precision_at_5:.4f}")
print(f"Recall@5: {recall_at_5:.4f}")
print(f"nDCG@5: {ndcg_at_5:.4f}")


SBERT Top 5 Results:
1. ds126 – AUSTRALIA – AUTOMOTIVE LEATHER II1
2. ds126A – AUSTRALIA – AUTOMOTIVE LEATHER II (ARTICLE 21.5 – US)1
3. ds367 – AUSTRALIA – APPLES1
4. ds18A – AUSTRALIA – SALMON (ARTICLE 21.5 – CANADA)1
5. ds529 – AUSTRALIA – ANTI-DUMPING MEASURES ON PAPER1

 SBERT Evaluation Metrics:
Precision@5: 1.0000
Recall@5: 1.0000
nDCG@5: 1.0000
